# Interactive embedding renderer

Explore cached MNIST-family and arXiv layouts with Datashader. This notebook can refit IMSTE and apply an optional gravity phase to an existing IMSTE layout. Run notebook 00 to create the MNIST cache; run `data/arxiv/prepare_arxiv_dataset.ipynb` and notebook 03 to prepare the arXiv vectors and coordinates.


In [1]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
sys.path.insert(0, str(PROJECT_ROOT))
import importlib
import imste.visualization as _imste_visualization
_imste_visualization = importlib.reload(_imste_visualization)
render_datashader_embedding = _imste_visualization.render_datashader_embedding

def render_embedding_preserving_aspect(embedding, labels=None, **kwargs):
    """Render without geometric stretching; support older IMSTE installs too."""
    import inspect
    options = dict(kwargs)
    renderer_parameters = inspect.signature(render_datashader_embedding).parameters
    if "preserve_aspect" in renderer_parameters:
        options["preserve_aspect"] = True
    else:
        coordinates = np.asarray(embedding)
        width = int(options.get("width", 1000))
        height = int(options.get("height", 1000))
        x_limits = np.asarray(
            options.get("x_range") or (coordinates[:, 0].min(), coordinates[:, 0].max()),
            dtype=np.float64,
        )
        y_limits = np.asarray(
            options.get("y_range") or (coordinates[:, 1].min(), coordinates[:, 1].max()),
            dtype=np.float64,
        )
        pixel_scale = max(
            (x_limits[1] - x_limits[0]) / width,
            (y_limits[1] - y_limits[0]) / height,
            np.finfo(np.float64).eps,
        )
        x_center, y_center = float(x_limits.mean()), float(y_limits.mean())
        options["x_range"] = (
            x_center - pixel_scale * width / 2,
            x_center + pixel_scale * width / 2,
        )
        options["y_range"] = (
            y_center - pixel_scale * height / 2,
            y_center + pixel_scale * height / 2,
        )
    return render_datashader_embedding(embedding, labels, **options)



## Load available coordinate caches

In [2]:
DATASETS = {}
DATASET_META = {}
mnist_path = PROJECT_ROOT / "data" / "embeddings" / "mnist_gallery_embeddings.npz"
if mnist_path.is_file():
    with np.load(mnist_path, allow_pickle=False) as saved:
        sample_count = int(saved["max_samples"].item()) if "max_samples" in saved else 60_000
        sample_seed = int(saved["random_state"].item()) if "random_state" in saved else 42
        for index, name in enumerate(saved["dataset_names"].astype(str)):
            DATASETS[name] = {
                "labels": saved[f"labels_{index}"],
                "UMAP": (saved[f"umap_{index}"], float(saved[f"umap_seconds_{index}"])),
                "IMSTE": (saved[f"imste_{index}"], float(saved[f"imste_seconds_{index}"])),
            }
            DATASET_META[name] = {
                "source": "mnist",
                "max_samples": sample_count,
                "sample_seed": sample_seed,
            }

arxiv_path = PROJECT_ROOT / "data" / "arxiv" / "cache" / "arxiv_embeddings.npz"
if arxiv_path.is_file():
    try:
        with np.load(arxiv_path, allow_pickle=False) as saved:
            labels = saved["labels"].astype(str)
            umap_embedding = saved["umap"]
            imste_embedding = saved["imste_multilevel"]
            umap_seconds = float(saved["umap_seconds"])
            imste_seconds = float(saved["imste_seconds"])
    except ValueError as error:
        if "Object arrays cannot be loaded" not in str(error):
            raise
        # Older notebook runs saved pandas string labels as object arrays.
        # This project-created cache is local; new caches use Unicode labels.
        with np.load(arxiv_path, allow_pickle=True) as saved:
            labels = saved["labels"].astype(str)
            umap_embedding = saved["umap"]
            imste_embedding = saved["imste_multilevel"]
            umap_seconds = float(saved["umap_seconds"])
            imste_seconds = float(saved["imste_seconds"])
    arxiv_name = f"arXiv ({len(labels):,})"
    DATASETS[arxiv_name] = {
        "labels": labels,
        "UMAP": (umap_embedding, umap_seconds),
        "IMSTE · multilevel initialization": (imste_embedding, imste_seconds),
    }
    DATASET_META[arxiv_name] = {"source": "arxiv"}

if not DATASETS:
    raise FileNotFoundError("No embedding caches found. Run notebook 00 for MNIST or data/arxiv/prepare_arxiv_dataset.ipynb and notebook 03 for arXiv.")
print("Available datasets:", ", ".join(DATASETS))


Available datasets: MNIST (28×28), Fashion-MNIST (28×28), Kuzushiji-MNIST (28×28), arXiv (39,906)


## Explore, refit, and apply gravity

Choose a dataset and layout, adjust the raster appearance, or open the unified IMSTE controls. The gravity second phase is a section alongside fit, graph, optimization, and initialization settings. Select an IMSTE layout, tune its parameters, and apply gravity; UMAP is never an input. Gravity uses feature-space RINND neighbors and Leiden regions, while labels are used only for rendering and the post-fit silhouette. Outputs stay in notebook memory and do not overwrite source caches.


In [ ]:
dataset_widget = widgets.Dropdown(options=list(DATASETS), description="Dataset:", layout=widgets.Layout(width="450px"))
method_widget = widgets.Dropdown(description="Layout:")
width_widget = widgets.IntSlider(value=700, min=400, max=2200, step=100, description="Width:", continuous_update=False)
height_widget = widgets.IntSlider(value=700, min=400, max=1800, step=100, description="Height:", continuous_update=False)
spread_widget = widgets.IntSlider(value=0, min=0, max=12, step=1, description="Spread px:", continuous_update=False)
alpha_widget = widgets.IntSlider(value=40, min=0, max=255, step=5, description="Min alpha:", continuous_update=False)
background_widget = widgets.Dropdown(options=["white", "black"], value="white", description="Background:")
render_button = widgets.Button(description="Render", button_style="primary", icon="image")
refit_button = widgets.Button(description="Refit IMSTE", button_style="warning", icon="refresh")
output = widgets.Output()
refit_output = widgets.Output()
gravity_output = widgets.Output()

# All estimator options that affect a fit are exposed below. For n_components > 2,
# the viewer displays the first two coordinates. Callback hooks are programmatic
# callbacks, not notebook fit settings. init_coordinates is a cached-layout warm start.
imste_widgets = {
    "n_components": widgets.IntSlider(value=2, min=2, max=8, step=1, description="Dimensions:", continuous_update=False),
    "n_msts": widgets.IntSlider(value=4, min=1, max=64, step=1, description="MST count:", continuous_update=False),
    "n_epochs": widgets.IntSlider(value=500, min=0, max=2000, step=50, description="Epochs:", continuous_update=False),
    "batch_size": widgets.IntSlider(value=32768, min=4096, max=131072, step=4096, description="Batch size:", continuous_update=False),
    "learning_rate": widgets.FloatSlider(value=0.05, min=0.001, max=0.2, step=0.001, readout_format=".3f", description="Learning rate:", continuous_update=False),
    "negative_ratio": widgets.IntSlider(value=1, min=0, max=30, step=1, description="Negative ratio:", continuous_update=False),
    "lambda_rep": widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, readout_format=".2f", description="Repulsion share (λ):", continuous_update=False),
    "random_state": widgets.IntSlider(value=42, min=0, max=999, step=1, description="Random seed:", continuous_update=False),
    "epsilon": widgets.FloatLogSlider(value=1e-4, base=10, min=-6, max=-2, step=0.1, readout_format=".1e", description="Epsilon:", continuous_update=False),
    "device": widgets.Dropdown(options=["auto", "cpu", "mps", "cuda"], value="mps", description="Device:"),
    "logistic_margin": widgets.FloatSlider(value=1.0, min=0.0, max=25.0, step=0.1, description="Logistic margin:", continuous_update=False),
    "logistic_temperature": widgets.FloatSlider(value=0.5, min=0.01, max=5.0, step=0.01, description="Logistic temperature:", continuous_update=False),
    "mst_method": widgets.Dropdown(options=["famst", "prim"], value="famst", description="MST builder:"),
    "mst_neighbors": widgets.IntSlider(value=15, min=1, max=256, step=1, description="MST neighbors:", continuous_update=False),
    "mst_inter_component_edges": widgets.IntSlider(value=5, min=1, max=50, step=1, description="Bridge edges:", continuous_update=False),
    "mst_max_neighbors": widgets.IntSlider(value=15, min=1, max=512, step=1, description="FAMST max neighbors:", continuous_update=False),
    "mst_representatives_per_component": widgets.IntSlider(value=10, min=1, max=100, step=1, description="MST reps / component:", continuous_update=False),
    "pca_init": widgets.Checkbox(value=True, description="PCA initialization"),
    "pca_init_scaling": widgets.Dropdown(options=["global", "none"], value="global", description="PCA scaling:"),
    "multilevel_init": widgets.Checkbox(value=True, description="Multilevel initialization"),
    "multilevel_reduction_ratio": widgets.IntSlider(value=4, min=2, max=16, step=1, description="Multilevel reduction:", continuous_update=False),
    "multilevel_base_size": widgets.IntSlider(value=1024, min=256, max=8192, step=256, description="Multilevel base size:", continuous_update=False),
    "multilevel_epochs": widgets.IntSlider(value=10, min=0, max=100, step=1, description="Multilevel epochs:", continuous_update=False),
    "metric": widgets.Dropdown(options=["euclidean", "cosine"], value="euclidean", description="Metric:"),
    "use_cached_init": widgets.Checkbox(value=False, description="Use cached IMSTE as initial coordinates"),
}
def _parameter_rows(names):
    items = []
    for name in names:
        item = imste_widgets[name]
        item.style.description_width = "190px"
        item.layout = widgets.Layout(width="680px")
        items.append(item)
    return items

parameter_panels = widgets.Accordion(children=[
    widgets.VBox(_parameter_rows([
        "n_components", "n_epochs", "batch_size", "learning_rate", "random_state", "device",
    ])),
    widgets.VBox(_parameter_rows([
        "metric", "n_msts", "mst_method", "mst_neighbors", "mst_max_neighbors",
        "mst_inter_component_edges", "mst_representatives_per_component",
    ])),
    widgets.VBox(_parameter_rows([
        "negative_ratio", "lambda_rep", "epsilon", "logistic_margin", "logistic_temperature",
    ])),
    widgets.VBox(_parameter_rows([
        "pca_init", "pca_init_scaling", "multilevel_init", "multilevel_reduction_ratio",
        "multilevel_base_size", "multilevel_epochs", "use_cached_init",
    ])),
])
parameter_panels.layout = widgets.Layout(width="720px")
for index, title in enumerate(["Embedding and device", "MST graph", "Optimization", "Initialization"]):
    parameter_panels.set_title(index, title)
parameter_note = widgets.HTML(
    "<small>For dimensions above 2, the viewer shows the first two coordinates. "
    "Progress callbacks are Python hooks. Refit results stay in memory and do not "
    "replace cache files.</small>"
)
refit_panel = widgets.VBox([
    widgets.HTML("<b>IMSTE fit and gravity settings</b>"), parameter_panels,
    parameter_note, refit_button, refit_output,
], layout=widgets.Layout(width="740px"))

# Gravity is a configurable second phase on an existing IMSTE layout only.
gravity_source_widget = widgets.Dropdown(description="IMSTE source:")
gravity_mode_widget = widgets.Dropdown(
    options=[("Mexican hat: local force", "mexican_hat"), ("Density-core pull", "density_core")],
    value="mexican_hat", description="Force mode:",
)
gravity_strength_widget = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, readout_format=".2f", description="Core pull strength:", continuous_update=False)
gravity_resolution_widget = widgets.FloatSlider(value=0.8, min=0.1, max=12.0, step=0.1, readout_format=".1f", description="Leiden resolution:", continuous_update=False)
gravity_core_widget = widgets.FloatSlider(value=0.95, min=0.50, max=0.99, step=0.01, readout_format=".2f", description="Core quantile:", continuous_update=False)
gravity_mass_power_widget = widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.25, readout_format=".2f", description="Mass exponent:", continuous_update=False)
gravity_neighbors_widget = widgets.IntSlider(value=15, min=5, max=64, step=1, description="Graph neighbors:", continuous_update=False)
gravity_attraction_widget = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05, readout_format=".2f", description="Local attraction:", continuous_update=False)
gravity_near_repulsion_widget = widgets.FloatSlider(value=1.5, min=0.0, max=3.0, step=0.05, readout_format=".2f", description="Near repulsion:", continuous_update=False)
gravity_force_neighbors_widget = widgets.IntSlider(value=8, min=2, max=32, step=1, description="2D force neighbors:", continuous_update=False)
gravity_attraction_range_widget = widgets.FloatSlider(value=1.0, min=0.25, max=3.0, step=0.05, readout_format=".2f", description="Attraction range:", continuous_update=False)
gravity_repulsion_range_widget = widgets.FloatSlider(value=0.2, min=0.05, max=1.0, step=0.05, readout_format=".2f", description="Local repulsion range:", continuous_update=False)
gravity_step_widget = widgets.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.005, readout_format=".3f", description="Force step size:", continuous_update=False)
gravity_iterations_widget = widgets.IntSlider(value=3, min=0, max=20, step=1, description="Force iterations:", continuous_update=False)
gravity_button = widgets.Button(description="Apply gravity phase", button_style="success", icon="magnet")
gravity_note = widgets.HTML(
    "<small>Core pull uses density-weighted Leiden regions. Mexican-hat interactions use only nearby 2D point pairs: "
    "moderate distances attract and very close pairs repel to prevent collapse. The ranges scale with each point's local neighbor radius. "
    "Labels are used only for the reported silhouette.</small>"
)
gravity_widgets = [
    gravity_source_widget, gravity_mode_widget, gravity_strength_widget,
    gravity_resolution_widget, gravity_core_widget, gravity_mass_power_widget,
    gravity_neighbors_widget, gravity_attraction_widget, gravity_near_repulsion_widget,
    gravity_force_neighbors_widget, gravity_attraction_range_widget,
    gravity_repulsion_range_widget, gravity_step_widget, gravity_iterations_widget,
]
for _gravity_widget in gravity_widgets:
    _gravity_widget.style.description_width = "190px"
    _gravity_widget.layout = widgets.Layout(width="680px")
gravity_controls = widgets.VBox([*gravity_widgets, gravity_note, gravity_button, gravity_output], layout=widgets.Layout(width="740px"))
parameter_panels.children = (*parameter_panels.children, gravity_controls)
parameter_panels.set_title(len(parameter_panels.children) - 1, "Gravity second phase")
parameter_panels.layout = widgets.Layout(width="740px")
fit_input_cache = {}
gravity_graph_cache = {}


def _get_gravity_regions(dataset_name, X, resolution, neighbors):
    """Return density weights and a Leiden partition of the feature-space RINND graph."""
    cache_key = (dataset_name, int(len(X)), float(resolution), int(neighbors), int(imste_widgets["random_state"].value))
    if cache_key in gravity_graph_cache:
        return gravity_graph_cache[cache_key]

    try:
        from rinnd import RINND
    except ImportError as error:
        raise ImportError("Gravity requires RINND. Install it with `python -m pip install 'imste[rinnd]'`.") from error
    try:
        import igraph as ig
        import leidenalg
    except ImportError as error:
        raise ImportError("Gravity region detection requires igraph and leidenalg. Install them with `python -m pip install igraph leidenalg`.") from error

    n_samples = len(X)
    if n_samples < 3:
        raise ValueError("Gravity requires at least three input rows.")
    neighbors = min(int(neighbors), n_samples - 1)
    features = np.ascontiguousarray(X, dtype=np.float32)
    started = time.perf_counter()
    index = RINND(
        features, metric="euclidean", n_neighbors=neighbors,
        random_state=int(imste_widgets["random_state"].value), n_trees=8,
        n_iters=max(5, int(round(np.log2(n_samples)))),
        max_candidates=max(20, min(60, int(1.5 * neighbors))),
        n_jobs=-1, verbose=False,
    )
    indices, distances = index.neighbor_graph
    indices = np.asarray(indices, dtype=np.int64)
    distances = np.asarray(distances, dtype=np.float32)
    radii = np.max(distances, axis=1)
    raw_mass = (np.median(radii) / np.maximum(radii, 1e-12)) ** 2
    node_mass = np.clip(raw_mass, np.percentile(raw_mass, 1), np.percentile(raw_mass, 99)).astype(np.float32)
    rows = np.repeat(np.arange(n_samples, dtype=np.int64), indices.shape[1])
    cols = indices.reshape(-1)
    edge_distances = distances.reshape(-1)
    valid = (cols >= 0) & (cols != rows) & np.isfinite(edge_distances)
    rows, cols, edge_distances = rows[valid], cols[valid], edge_distances[valid]
    edge_weights = np.exp(-edge_distances / np.maximum(radii[rows], 1e-12)).astype(np.float32)
    graph = ig.Graph(n=n_samples, edges=zip(rows.tolist(), cols.tolist()), directed=False)
    graph.es["weight"] = edge_weights.tolist()
    graph.simplify(multiple=True, loops=True, combine_edges={"weight": "sum"})
    partition = leidenalg.find_partition(
        graph, leidenalg.RBConfigurationVertexPartition, weights="weight",
        resolution_parameter=float(resolution),
        seed=int(imste_widgets["random_state"].value), n_iterations=2,
    )
    membership = np.asarray(partition.membership, dtype=np.int32)
    region_count = int(membership.max()) + 1
    region_mass = np.bincount(membership, weights=node_mass, minlength=region_count)
    graph_seconds = time.perf_counter() - started
    result = (node_mass, membership, region_mass, graph_seconds)
    gravity_graph_cache[cache_key] = result
    return result


def _apply_gravity_to_layout(coordinates, node_mass, membership, region_mass, strength, core_quantile, mass_power):
    """Pull each point toward its region's density-weighted core."""
    n_regions = len(region_mass)
    order = np.argsort(membership, kind="stable")
    sorted_membership = membership[order]
    sorted_mass = node_mass[order]
    boundaries = np.r_[0, np.flatnonzero(np.diff(sorted_membership)) + 1, len(order)]
    thresholds = np.empty(n_regions, dtype=np.float32)
    for start, stop in zip(boundaries[:-1], boundaries[1:]):
        thresholds[sorted_membership[start]] = np.quantile(sorted_mass[start:stop], core_quantile)
    core = node_mass >= thresholds[membership]
    core_mass = np.bincount(membership[core], weights=node_mass[core], minlength=n_regions)
    centers = np.zeros((n_regions, coordinates.shape[1]), dtype=np.float64)
    np.add.at(
        centers, membership[core],
        np.asarray(coordinates, dtype=np.float64)[core] * node_mass[core, None],
    )
    missing = core_mass <= 0
    if np.any(missing):
        for region in np.flatnonzero(missing):
            members = np.flatnonzero(membership == region)
            point = members[np.argmax(node_mass[members])]
            centers[region] = coordinates[point]
            core_mass[region] = 1.0
    centers /= core_mass[:, None]
    region_strength = np.clip(
        strength * (region_mass / np.median(region_mass)) ** mass_power, 0.0, 0.9
    )
    result = np.asarray(coordinates, dtype=np.float64).copy()
    result += region_strength[membership, None] * (centers[membership] - result)
    result -= result.mean(axis=0)
    return result.astype(np.float32), n_regions, float(np.mean(core))


def update_methods(*_):
    layouts = [key for key in DATASETS[dataset_widget.value] if key != "labels"]
    method_widget.options = layouts
    method_widget.value = layouts[0]
    gravity_layouts = [key for key in layouts if key.startswith("IMSTE")]
    gravity_source_widget.options = gravity_layouts
    if gravity_layouts:
        gravity_source_widget.value = gravity_layouts[0]


def render_selected(_=None):
    selected = DATASETS[dataset_widget.value]
    embedding, seconds = selected[method_widget.value]
    visible_coordinates = embedding[:, :2]
    image = render_embedding_preserving_aspect(
        visible_coordinates,
        selected["labels"],
        width=width_widget.value,
        height=height_widget.value,
        spread_px=spread_widget.value,
        min_alpha=alpha_widget.value,
        background=background_widget.value,
    )
    with output:
        clear_output(wait=True)
        figure_width = 4
        figure_height = figure_width * height_widget.value / width_widget.value
        fig, ax = plt.subplots(figsize=(figure_width, figure_height), constrained_layout=True)
        ax.imshow(image.to_pil())
        runtime = f"{seconds / 60:.1f} min" if seconds >= 60 else f"{seconds:.1f} s"
        component_note = f" (first 2 of {embedding.shape[1]})" if embedding.shape[1] > 2 else ""
        ax.set_title(
            f"{dataset_widget.value} — {method_widget.value}{component_note} (fit: {runtime})",
            fontsize=6,
        )
        ax.set_axis_off()
        display(fig)
        plt.close(fig)


def apply_gravity_selected(_=None):
    with gravity_output:
        clear_output(wait=True)
        dataset_name = dataset_widget.value
        source_name = gravity_source_widget.value
        if not source_name or not source_name.startswith("IMSTE"):
            print("Select an IMSTE layout as the gravity input; UMAP layouts are not eligible.")
            return
        try:
            X, labels = _load_refit_input(dataset_name)
            coordinates, base_seconds = DATASETS[dataset_name][source_name]
            strength = float(gravity_strength_widget.value)
            resolution = float(gravity_resolution_widget.value)
            core_quantile = float(gravity_core_widget.value)
            mass_power = float(gravity_mass_power_widget.value)
            mode = gravity_mode_widget.value
            graph_neighbors = int(gravity_neighbors_widget.value)
            attraction_strength = float(gravity_attraction_widget.value)
            near_repulsion_strength = float(gravity_near_repulsion_widget.value)
            force_neighbors = int(gravity_force_neighbors_widget.value)
            attraction_range = float(gravity_attraction_range_widget.value)
            repulsion_range = float(gravity_repulsion_range_widget.value)
            step_size = float(gravity_step_widget.value)
            iterations = int(gravity_iterations_widget.value)
            if repulsion_range >= attraction_range:
                raise ValueError("Local repulsion range must be smaller than attraction range.")
            print(f"Gravity second phase on {dataset_name}: {len(X):,} rows", flush=True)
            started = time.perf_counter()
            if strength == 0.0 and mode == "density_core" and iterations == 0:
                gravity_embedding = np.asarray(coordinates, dtype=np.float32).copy()
                region_count, core_fraction, graph_seconds = 0, 0.0, 0.0
            else:
                print("Building/reusing RINND graph and Leiden regions…", flush=True)
                node_mass, membership, region_mass, graph_seconds = _get_gravity_regions(
                    dataset_name, X, resolution, graph_neighbors
                )
                print(f"Applying {mode} force to {len(region_mass):,} regions…", flush=True)
                gravity_embedding, region_count, core_fraction = _apply_gravity_to_layout(
                    coordinates, node_mass, membership, region_mass,
                    strength, core_quantile, mass_power,
                )
                if mode == "mexican_hat":
                    from arxiv_imste import apply_local_mexican_hat_force
                    gravity_embedding = apply_local_mexican_hat_force(
                        gravity_embedding, node_mass,
                        attraction_strength=attraction_strength,
                        near_repulsion_strength=near_repulsion_strength,
                        attraction_range=attraction_range,
                        repulsion_range=repulsion_range,
                        neighbors=force_neighbors,
                        step_size=step_size,
                        iterations=iterations,
                        mass_power=mass_power,
                    )
            gravity_seconds = time.perf_counter() - started
            from sklearn.metrics import silhouette_score
            silhouette = silhouette_score(
                gravity_embedding, np.asarray(labels),
                sample_size=min(5_000, len(labels)), random_state=2027,
            )
            result_name = (
                f"IMSTE · gravity {mode} (g={strength:g}, r={resolution:g}, "
                f"core={core_quantile:.2f}, local={attraction_strength:g}/{near_repulsion_strength:g})"
            )
            DATASETS[dataset_name][result_name] = (
                gravity_embedding, float(base_seconds + gravity_seconds)
            )
            update_methods()
            method_widget.value = result_name
            gravity_source_widget.value = result_name
            print(f"Regions: {region_count:,}; core fraction: {core_fraction:.1%}")
            if mode == "mexican_hat":
                print(
                    f"Local force: {attraction_strength:g} attraction / {near_repulsion_strength:g} near repulsion; "
                    f"ranges {attraction_range:g}/{repulsion_range:g}; {force_neighbors} neighbors × {iterations} iterations"
                )
            print(f"Graph/partition: {graph_seconds:.1f}s; gravity phase: {gravity_seconds:.1f}s")
            print(f"Label silhouette (sample ≤5,000): {silhouette:.4f}")
            print("Labels were not used in graph construction, force calculation, or initialization.")
            render_selected()
        except Exception as error:
            print(f"Gravity phase failed: {type(error).__name__}: {error}")


def _load_refit_input(dataset_name):
    if dataset_name in fit_input_cache:
        return fit_input_cache[dataset_name]

    meta = DATASET_META[dataset_name]
    if meta["source"] == "arxiv":
        vector_path = PROJECT_ROOT / "data" / "arxiv" / "cache" / "arxiv_vectors.npy"
        if not vector_path.is_file():
            raise FileNotFoundError(f"Prepared arXiv vectors are missing: {vector_path}")
        X = np.load(vector_path, mmap_mode="r")
        labels = DATASETS[dataset_name]["labels"]
    else:
        from high_dim_mst_gallery import _load_kmnist, _load_openml, _stratified_sample
        from sklearn.preprocessing import StandardScaler

        if dataset_name == "MNIST (28×28)":
            raw_X, raw_labels = _load_openml("mnist_784")
        elif dataset_name == "Fashion-MNIST (28×28)":
            raw_X, raw_labels = _load_openml("Fashion-MNIST")
        elif dataset_name == "Kuzushiji-MNIST (28×28)":
            raw_X, raw_labels = _load_kmnist()
        else:
            raise ValueError(f"No high-dimensional source is configured for {dataset_name!r}.")
        X, labels = _stratified_sample(
            raw_X,
            raw_labels,
            max_samples=meta["max_samples"],
            random_state=meta["sample_seed"],
        )
        X = StandardScaler().fit_transform(np.asarray(X, dtype=np.float32))

    if len(X) != len(DATASETS[dataset_name]["labels"]):
        raise ValueError("Source data and cached layouts have different row counts.")
    fit_input_cache[dataset_name] = (X, labels)
    return fit_input_cache[dataset_name]


def refit_selected_imste(_=None):
    with refit_output:
        clear_output(wait=True)
        dataset_name = dataset_widget.value
        print(f"Preparing input for {dataset_name}…", flush=True)
        try:
            X, fit_labels = _load_refit_input(dataset_name)
            if len(fit_labels) != len(DATASETS[dataset_name]["labels"]):
                raise ValueError("Refit labels do not align with the cached layouts.")
            from imste import IteratedMinimumSpanningTreeEmbedder
            import time

            settings = {name: widget.value for name, widget in imste_widgets.items() if name != "use_cached_init"}
            settings["init_coordinates"] = None
            if imste_widgets["use_cached_init"].value:
                if settings["n_components"] != 2:
                    raise ValueError("Cached coordinates can initialize a 2D refit only.")
                cached_names = [key for key in DATASETS[dataset_name] if key.startswith("IMSTE")]
                if not cached_names:
                    raise ValueError("No cached IMSTE layout is available for initialization.")
                settings["init_coordinates"] = DATASETS[dataset_name][cached_names[0]][0]
            print(f"Fitting IMSTE with {len(X):,} samples × {X.shape[1]:,} features…", flush=True)
            started = time.perf_counter()
            refit_embedding = IteratedMinimumSpanningTreeEmbedder(**settings).fit_transform(X)
            seconds = time.perf_counter() - started
            result_name = f"IMSTE · refit ({seconds:.1f}s)"
            DATASETS[dataset_name][result_name] = (refit_embedding, seconds)
            previous_name = result_name
            update_methods()
            method_widget.value = previous_name
            gravity_source_widget.value = previous_name
            print(f"Finished in {seconds / 60:.1f} min" if seconds >= 60 else f"Finished in {seconds:.1f} s")
            render_selected()
        except Exception as error:
            print(f"Refit failed: {type(error).__name__}: {error}")


dataset_widget.observe(update_methods, names="value")
render_button.on_click(render_selected)
refit_button.on_click(refit_selected_imste)
gravity_button.on_click(apply_gravity_selected)
update_methods()
display(widgets.VBox([
    dataset_widget, method_widget,
    widgets.HBox([width_widget, height_widget]),
    widgets.HBox([spread_widget, alpha_widget]),
    widgets.HBox([background_widget, render_button]),
    refit_panel,
    output,
]))
render_selected()
